In [0]:
%sql
DESCRIBE DETAIL ecomotive.gold.dim_drivers;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,4144c62d-1aa7-4de3-a879-a2c25c865437,ecomotive.gold.dim_drivers,null,,2025-12-19T14:14:21.461Z,2026-01-12T10:48:02.000Z,List(),List(),1,5183,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.workloadBasedColumns.optimizerStatistics -> `driver_id`, delta.writePartitionColumnsToParquet -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-17aee899-91a2-4877-a45e-a7d2d891cc5d, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-f1e752d8-956f-45d0-9177-dc265bb0fa60)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Cambiamos la estrategia de la tabla a Liquid Clustering
ALTER TABLE ecomotive.gold.dim_drivers
CLUSTER BY (driver_id);

In [0]:
# Leemos tu tabla
df = spark.read.table("ecomotive.gold.dim_drivers")

# --- EL SABOTAJE ---
# Reparticionamos los datos en 15 archivos (¡Más archivos que filas!)
# Esto simula una tabla que ha recibido datos gota a gota durante meses.
df.repartition(15).write.mode("overwrite").saveAsTable("ecomotive.gold.dim_drivers")

print("✅ Tabla fragmentada en muchos archivos pequeños.")

✅ Tabla fragmentada en muchos archivos pequeños.


In [0]:
# 1. Copia de seguridad REAL (Materializada en otra tabla temporal)
# Así rompemos el vínculo con la tabla original antes de borrarla.
spark.read.table("ecomotive.gold.dim_drivers").write.mode("overwrite").saveAsTable("temp_drivers_backup")

# 2. Ahora sí, borramos la original sin miedo
spark.sql("TRUNCATE TABLE ecomotive.gold.dim_drivers")

# 3. Preparamos la fuente leyendo de la COPIA
df_source = spark.read.table("temp_drivers_backup")

# 4. EL SABOTAJE (Ingesta Gota a Gota)
print("Inicio del sabotaje...")
for i in range(0, 10): 
    # Escribimos los datos 10 veces
    df_source.write.mode("append").saveAsTable("ecomotive.gold.dim_drivers")
    print(f"Ingesta {i+1} completada (Archivo creado).")

# 5. Limpieza de la tabla temporal
spark.sql("DROP TABLE temp_drivers_backup")

print("✅ Sabotaje completado. La tabla está fragmentada.")

Inicio del sabotaje...
Ingesta 1 completada (Archivo creado).
Ingesta 2 completada (Archivo creado).
Ingesta 3 completada (Archivo creado).
Ingesta 4 completada (Archivo creado).
Ingesta 5 completada (Archivo creado).
Ingesta 6 completada (Archivo creado).
Ingesta 7 completada (Archivo creado).
Ingesta 8 completada (Archivo creado).
Ingesta 9 completada (Archivo creado).
Ingesta 10 completada (Archivo creado).
✅ Sabotaje completado. La tabla está fragmentada.


In [0]:
%sql
DESCRIBE DETAIL ecomotive.gold.dim_drivers;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,4144c62d-1aa7-4de3-a879-a2c25c865437,ecomotive.gold.dim_drivers,null,,2025-12-19T14:14:21.461Z,2026-01-12T10:52:10.000Z,List(),List(driver_id),10,64900,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.workloadBasedColumns.optimizerStatistics -> `driver_id`, delta.writePartitionColumnsToParquet -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-17aee899-91a2-4877-a45e-a7d2d891cc5d, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-f1e752d8-956f-45d0-9177-dc265bb0fa60)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Fíjate que NO ponemos ZORDER BY. Ya está definido en la tabla.
OPTIMIZE ecomotive.gold.dim_drivers;

path,metrics
,"List(1, 10, List(8404, 8404, 8404.0, 1, 8404), List(6490, 6490, 6490.0, 10, 64900), 0, null, null, 0, 1, 10, 0, false, 0, 0, 1768215148924, 1768215155928, 8, 1, null, List(0, 0), null, 6, 6, 845, 0, List(64900, true, false, false, 0.0, List(0.0), 1.0, null, 0, 0, 0, 0, 10, 64900, 64900, null, log, 16777216, 67108864, 4, 0, 0, List(0), 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 64900, 0, 64900, 0, 0, 0, 0, 0, 64900, 64900, List(74, 62, 602, 54, 0, 1011), 2, 1, 5, sizeAware))"
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, false, 0, 0, 1768215155975, 1768215157723, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, List(8404, false, false, false, 0.0, List(0.0), 1.0, null, 0, 0, 0, 0, 0, 0, 0, null, log, 16777216, 67108864, 4, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(46, 5, 443, 0, 0, 0), 2, 2, 5, sizeAware))"
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, true, 0, 0, 1768215157777, 1768215158690, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, List(8404, false, false, false, 0.0, List(0.0), 1.0, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 418, 0, 0, 0), 15, 1, 1, null))"


In [0]:
%sql
DESCRIBE DETAIL ecomotive.gold.dim_drivers;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,4144c62d-1aa7-4de3-a879-a2c25c865437,ecomotive.gold.dim_drivers,null,,2025-12-19T14:14:21.461Z,2026-01-12T10:52:36.000Z,List(),List(driver_id),1,8404,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.workloadBasedColumns.optimizerStatistics -> `driver_id`, delta.writePartitionColumnsToParquet -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-17aee899-91a2-4877-a45e-a7d2d891cc5d, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-f1e752d8-956f-45d0-9177-dc265bb0fa60)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
